In [ ]:
# !pip install mamba-ssm --no-build-isolation
# ensure all requirements are downloaded

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def calc_si_sdr(estimate, target, epsilon=1e-8):
    """
    Calculates Negative Scale-Invariant Signal-to-Distortion Ratio.
    Shapes expected: (batch_size, sequence_length)
    """
    target = target - torch.mean(target, dim=-1, keepdim=True)
    estimate = estimate - torch.mean(estimate, dim=-1, keepdim=True)
    
    target_energy = torch.sum(target ** 2, dim=-1, keepdim=True) + epsilon
    dot_product = torch.sum(target * estimate, dim=-1, keepdim=True)
    alpha = dot_product / target_energy
    
    target_scaled = alpha * target
    noise = estimate - target_scaled
    
    signal_energy = torch.sum(target_scaled ** 2, dim=-1) + epsilon
    noise_energy = torch.sum(noise ** 2, dim=-1) + epsilon
    
    # Prevent log10 from receiving 0 or negative numbers
    ratio = signal_energy / noise_energy
    ratio = torch.clamp(ratio, min=epsilon)
    
    si_sdr = 10 * torch.log10(ratio)
    return -si_sdr

In [ ]:
def or_pit_loss(est_speaker, est_residual, target_sources):
    """
    Calculates the One-and-Rest PIT loss.
    est_speaker: (batch, length) - The single voice extracted
    est_residual: (batch, length) - The leftover audio
    target_sources: (batch, num_speakers, length) - All ground truth voices
    """
    batch_size, num_speakers, _ = target_sources.shape
    losses = []
    
    # Iterate through each potential speaker to find the best match
    for i in range(num_speakers):
        # The target for the single speaker
        target_spk = target_sources[:, i, :]
        
        # The target for the residual (sum of everyone except speaker i)
        # We create a mask to exclude the i-th speaker
        mask = torch.arange(num_speakers, device=target_sources.device) != i
        target_res = torch.sum(target_sources[:, mask, :], dim=1)
        
        # Calculate SI-SDR for both the extracted speaker and the residual pile
        loss_spk = calc_si_sdr(est_speaker, target_spk)
        loss_res = calc_si_sdr(est_residual, target_res)
        
        # Total loss for this specific permutation assumption
        total_loss = loss_spk + loss_res
        losses.append(total_loss)
        
    # Stack losses to shape: (batch_size, num_speakers)
    losses = torch.stack(losses, dim=1)
    
    # Find the minimum loss across the speaker dimension for each batch item
    min_loss, best_speaker_idx = torch.min(losses, dim=1)
    
    # Return the mean batch loss to update weights, and the index of who was extracted
    return min_loss.mean(), best_speaker_idx

In [ ]:
from mamba_ssm import Mamba

class MambaDeflationaryExtractor(nn.Module):
    def __init__(self, d_model=256, d_state=16, d_conv=4, expand=2, num_blocks=4):
        super().__init__()
        
        # Encoder: Audio waveform to latent features (1D Conv)
        self.encoder = nn.Conv1d(in_channels=1, out_channels=d_model, kernel_size=16, stride=8, padding=4)
        
        # Backbone: Stacked Mamba Blocks (Ultra-efficient sequential processing)
        self.mamba_blocks = nn.ModuleList([
            Mamba(
                d_model=d_model, # Model dimension
                d_state=d_state, # State dimension (controls memory compression)
                d_conv=d_conv,   # Local convolution width
                expand=expand    # Block expansion factor
            ) for _ in range(num_blocks)
        ])
        
        # Dual Decoders: 
        # Decoder A isolates the dominant speaker
        self.speaker_decoder = nn.ConvTranspose1d(in_channels=d_model, out_channels=1, kernel_size=16, stride=8, padding=4)
        # Decoder B outputs the rest of the mixture
        self.residual_decoder = nn.ConvTranspose1d(in_channels=d_model, out_channels=1, kernel_size=16, stride=8, padding=4)

    def forward(self, x):
        """
        x shape: (batch_size, 1, sequence_length)
        """
        # Encode
        encoded = self.encoder(x) # Shape: (batch, d_model, seq_len_latent)
        
        # Mamba expects shape (batch, seq_len, d_model), so we transpose
        latent = encoded.transpose(1, 2)
        
        # Pass through Mamba backbone
        for block in self.mamba_blocks:
            latent = block(latent)
            
        # Transpose back for decoding
        latent = latent.transpose(1, 2) # Shape: (batch, d_model, seq_len_latent)
        
        # Decode into two separate audio signals
        est_speaker = self.speaker_decoder(latent)
        est_residual = self.residual_decoder(latent)
        
        # Squeeze out the channel dimension for loss calculation
        return est_speaker.squeeze(1), est_residual.squeeze(1)

In [ ]:
# dataset with automatic padding 
import os
import torch
import torchaudio
from torch.utils.data import Dataset

class CustomSpeechMixtureDataset(Dataset):
    def __init__(self, base_dir, num_speakers, fixed_length=80000): 
        # fixed_length: 80000 is 5 seconds at 16kHz. 
        self.base_dir = base_dir
        self.num_speakers = num_speakers
        self.fixed_length = fixed_length
        
        self.mix_folders = sorted([
            f for f in os.listdir(base_dir) 
            if os.path.isdir(os.path.join(base_dir, f)) and f.startswith('mix_')
        ])

    def __len__(self):
        return len(self.mix_folders)

    def __getitem__(self, idx):
        folder_name = self.mix_folders[idx]
        folder_path = os.path.join(self.base_dir, folder_name)
        
        # 1. Load and Pad/Truncate Mixture
        mix_path = os.path.join(folder_path, "mix.wav")
        mixture, _ = torchaudio.load(mix_path)
        mixture = self._pad_or_truncate(mixture)
        
        # 2. Load and Pad/Truncate Sources
        sources = []
        for i in range(1, self.num_speakers + 1):
            source_path = os.path.join(folder_path, f"s{i}.wav")
            source_audio, _ = torchaudio.load(source_path)
            source_audio = self._pad_or_truncate(source_audio)
            sources.append(source_audio)
            
        target_sources = torch.cat(sources, dim=0)
        return mixture, target_sources

    def _pad_or_truncate(self, audio):
        """Forces audio to be exactly fixed_length."""
        length = audio.shape[-1]
        if length > self.fixed_length:
            return audio[:, :self.fixed_length]  # Truncate
        else:
            padding = self.fixed_length - length
            return torch.nn.functional.pad(audio, (0, padding))  # Pad with zeros

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

def train_model(model, data_dir, num_speakers, epochs, batch_size=8, learning_rate=5e-5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Prepare Data using dynamic variables
    train_dataset = CustomSpeechMixtureDataset(
        base_dir=data_dir,
        num_speakers=num_speakers
    )
    
    # Fast parallel data loading
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    
    # 2. Setup Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
    
    # 3. The Epoch Loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        
        for batch_idx, (mixture, targets) in enumerate(train_loader):
            mixture = mixture.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            est_speaker, est_residual = model(mixture)
            
            # OR-PIT Loss
            loss, extracted_idx = or_pit_loss(est_speaker, est_residual, targets)
            
            # the NaN sheild
            if torch.isnan(loss) or torch.isinf(loss):
                print(f" Warning: Corrupted audio math at Batch {batch_idx}. Skipping to protect weights.")
                optimizer.zero_grad()
                continue
           
            
            # Backpropagation & Clipping
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            
            if batch_idx % 50 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(train_loader)} | SI-SDR Loss: {loss.item():.4f}")
                
        avg_loss = total_loss / len(train_loader)
        print(f"=== Epoch {epoch+1} Complete | Average Loss: {avg_loss:.4f} ===")
        
        # Save checkpoints dynamically based on the current speaker count phase
        if (epoch + 1) % 10 == 0 or (epoch + 1) == epochs:
            state_dict_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state_dict_to_save, f"mamba_dexformer_{num_speakers}spk_epoch_{epoch+1}.pth")
            
    # Return the trained model to pass it to the next phase
    return model

In [ ]:
import os
import torch
import torch.nn as nn

def training_pipeline():
    # 1. setup model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Global Device Setup: {device}")
    
    # Initialize a fresh model from scratch
    model = MambaDeflationaryExtractor(d_model=256, d_state=16, d_conv=4, expand=2, num_blocks=4)
    

    
    if torch.cuda.device_count() > 1:
        print(f"Activating {torch.cuda.device_count()} GPUs for the full curriculum")
        model = nn.DataParallel(model)
    model.to(device)

    # 2. curriculum config 
    root_dataset_dir = "../data/"
    curriculum = [
        {"speakers": 2, "folder": "train_mix2/", "epochs": 40}, 
        {"speakers": 3, "folder": "train_mix3/", "epochs": 25}, 
        {"speakers": 4, "folder": "train_mix4/", "epochs": 25},
        {"speakers": 5, "folder": "train_mix5/", "epochs": 25}
    ]
    
    # 3. execute training
    for phase in curriculum:
        spk_count = phase["speakers"]
        data_path = os.path.join(root_dataset_dir, phase["folder"])
        epochs = phase["epochs"]
        
        print(f"\n{'='*60}")
        print(f"Starting Phase: {spk_count}-SPEAKER CURRICULUM ({epochs} Epochs)")
        print(f"{'='*60}\n")
        
        model = train_model(
            model=model,
            data_dir=data_path, 
            num_speakers=spk_count, 
            epochs=epochs, 
            batch_size=8, 
            learning_rate=5e-5 
        )
        
        print(f"Phase {spk_count} Complete. Weights saved.")

# Trigger the full run
training_pipeline()